# Perform binary image classification with logistic regression


 - Author: Elwin van 't Wout
 - Affiliation: Pontificia Universidad Católica de Chile
 - Course: IMT2112
 - Date: 27-10-2023

This notebook provides an example implementation of binary image classification with logistic regression. The machine learning task involves the classification of images as one of two possible labels. The logistic regression can be viewed as a neural network with an input and output layer, using the sigmoid as activation function and a binary classification loss function.

In [ ]:
import numpy as np
import tensorflow_datasets as tfds
from matplotlib import pylab as plt

## Download the MNIST dataset

Let us download a labelled dataset with handwritten numbers, available from the TensorFlow library and called the MNIST dataset. The first time the dataset is used, it is downloaded from the internet and stored locally. Subsequent calls will load the dataset from the hard disk.

In [ ]:
(mnist_data_train, mnist_data_test), data_info = tfds.load("mnist", split=["train", "test"], with_info=True, download=True)

In [ ]:
print(data_info)

In [ ]:
fig = tfds.show_examples(mnist_data_train, data_info)

In [ ]:
print("Number of training data items:", data_info.splits["train"].num_examples)
print("Number of testing data items:", data_info.splits["test"].num_examples)

## Preprocess the dataset for binary classification

As shown above, the dataset has thousands of images of handwritten numbers with the correct label. Our objective is to find out if the handwritten number is odd or even. This is a binary classification problem that can be solved with logistic regression.

Let us create True and False labels for odd and even numbers, respectively, and convert the images to Numpy arrays.

In [ ]:
labels_true = [0, 2, 4, 6, 8]
labels_false = [1, 3, 5, 7, 9]

In [ ]:
train_dataset_images = [example['image'] for example in tfds.as_numpy(mnist_data_train)]
train_dataset_labels = [example['label'] in labels_true for example in tfds.as_numpy(mnist_data_train)]

test_dataset_images = [example['image'] for example in tfds.as_numpy(mnist_data_test)]
test_dataset_labels = [example['label'] in labels_true for example in tfds.as_numpy(mnist_data_test)]

In [ ]:
for n in range(3):
    fig = plt.imshow(train_dataset_images[n].squeeze(), cmap='gray')
    plt.title("label: "+str(train_dataset_labels[n]))
    plt.show()

Implementing logistic regression is easier when the images are converted into vectors with normalized values. Hence, let us reshape the image and normalize the color values between zero and one. Then, stack all images into a feature matrix of size (dimension, n_data) and stack all labels into a vector of size (1, n_data).

In [ ]:
train_dataset_images_vector = [image.flatten()/255 for image in train_dataset_images]

test_dataset_images_vector = [image.flatten()/255 for image in test_dataset_images]

In [ ]:
train_features = np.vstack(train_dataset_images_vector).T
train_labels = np.vstack(train_dataset_labels).T

test_features = np.vstack(test_dataset_images_vector).T
test_labels = np.vstack(test_dataset_labels).T

In [ ]:
dim, n_train = train_features.shape
print("The", n_train, "training data items have a dimension of", dim, "and dtype", train_features.dtype)

dim, n_test = test_features.shape
print("The", n_test, "testing data items have a dimension of", dim, "and dtype", test_features.dtype)

## Implementation of the logistic regression with NumPy

For logistic regression, we need to choose the learning rate for the gradient descent optimization, and initialize the weights and biases as all zeros.

In [ ]:
learning_rate = 0.01

In [ ]:
initial_weights = np.zeros([dim, 1], dtype=float)
initial_bias = 0.0

Define all functionality of logistic regression, such as the forward and backward pass, the update step of the parameters, and the training and prediction algorithms. The Numpy functions are vectorized and use multithreading by default.

In [ ]:
def sigmoid(z):
    return 1.0/(1.0+np.exp(-z))

def forward_propagation(x, w, b):
    z = w.T @ x + b
    a = sigmoid(z)
    return a

def backward_propagation(x, a, y):
    m = x.shape[1]
    j = -np.sum(y * np.log(a) + (1-y) * np.log(1-a)) / m
    dz = a - y
    dw = (x @ dz.T) / m
    db = np.mean(dz)
    return dw, db, j

def update(w, b, dw, db, lr):
    w -= lr * dw
    b -= lr * db
    return w, b

def train(x, y, w, b, lr, n_iter):
    costs = []
    for k in range(n_iter):
        a = forward_propagation(x, w, b)
        dw, db, j = backward_propagation(x, a, y)
        w, b = update(w, b, dw, db, lr)
        costs.append(j)
    return w, b, costs

def predict(x, w, b):
    a = forward_propagation(x, w, b)
    return np.round(a)


## Train the model

Let us train logistic regression on the image dataset, with a fixed number of iterations.

In [ ]:
n_iterations = 1000

In [ ]:
%%time
weights, bias, cost = train(train_features, train_labels, initial_weights, initial_bias, learning_rate, n_iterations)

In [ ]:
fig = plt.plot(cost)
plt.xlabel('iteration')
plt.ylabel('cost function')
plt.show()

## Perform the prediction

Let us check the accuracy by comparing the predictions with the actual labels, for both the training and testing data set.

In [ ]:
%%time
prediction_train = predict(train_features, weights, bias)

In [ ]:
%%time
prediction_test = predict(test_features, weights, bias)

In [ ]:
correct_train = np.sum(prediction_train == train_labels)
correct_test = np.sum(prediction_test == test_labels)

print("Predicted", correct_train, "items correctly from the", n_train, "training examples:", correct_train/n_train*100, "%")
print("Predicted", correct_test, "items correctly from the", n_test, "testing examples:", correct_test/n_test*100, "%")